In [1]:
import os
import glob
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import time

# **Generate Model Architecture & Initialize weights**

In [6]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import (
    Input, Conv2D, MaxPooling2D, MaxPooling1D, Flatten, LSTM, ConvLSTM2D , Dense, TimeDistributed, Dropout,
    GlobalAveragePooling2D, Concatenate, AveragePooling2D, BatchNormalization, Average,
    Reshape, AveragePooling3D, MaxPooling3D, GlobalMaxPooling3D
)

In [3]:
def build_1PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer")
    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
    ], name="CNN_Module")


    x = TimeDistributed(cnn, name="Time_Distributed_CNN")(in1)
    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=in1, outputs=x, name="1PTCSL")
    model.summary()

    return model

In [8]:
model1 = build_1PTCSL(input_shape=(10,256,256,3))
model1.load_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")
model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "1PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 10, 256, 256,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Time_Distributed_CNN            │ (None, 10, 32, 32,     │        93,248 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ConvLSTM_Layer (ConvLSTM2D)     │ (None, 10, 32, 32, 64) │       442,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling3d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling3D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten_Layer (Flatten)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Layer_1 (Dense)           │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
def build_2PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer_1")
    in2 = Input(shape=input_shape, name="Input_Layer_2")

    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2))
    ], name="CNN_Module")

    x1 = TimeDistributed(cnn, name="Time_Distributed_CNN_1")(in1)
    x2 = TimeDistributed(cnn, name="Time_Distributed_CNN_2")(in2)

    x = Average(name="Averaging_Layer")([x1, x2])

    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=[in1, in2], outputs=x, name="2PTCSL")
    model.summary()
    return model

In [27]:
model2 = build_2PTCSL(input_shape=(10,256,256,3))
model2.load_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/best_model.weights.h5")
model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "2PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
def build_3PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer_1")
    in2 = Input(shape=input_shape, name="Input_Layer_2")
    in3 = Input(shape=input_shape, name="Input_Layer_3")

    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
    ], name="CNN_Module")

    x1 = TimeDistributed(cnn, name="Time_Distributed_CNN_1")(in1)
    x2 = TimeDistributed(cnn, name="Time_Distributed_CNN_2")(in2)
    x3 = TimeDistributed(cnn, name="Time_Distributed_CNN_3")(in3)

    x = Average(name="Averaging_Layer")([x1, x2, x3])

    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=[in1, in2, in3], outputs=x, name="3PTCSL")
    model.summary()
    return model

In [20]:
model3 = build_3PTCSL(input_shape=(10,256,256,3))
model3.load_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/best_model.weights.h5")
model3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "3PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_3       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_3[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
│                     │                   │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

# **Test Stage**

In [14]:
test_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Test/"

In [22]:
loss, accuracy, precision, recall, auc = list(), list(), list(), list(), list()
for i in range(0,6):
      test_X0 = np.load(test_dir+f"Batch{i}X0.npy")
      test_X1 = np.load(test_dir+f"Batch{i}X1.npy")
      test_X2 = np.load(test_dir+f"Batch{i}X2.npy")
      test_y = np.load(test_dir+f"Batch{i}y.npy")
      print(f"Test Batch {i} of 6")
      test_metrics = model1.evaluate(test_X0, test_y)
      loss.append(test_metrics[0])
      accuracy.append(test_metrics[1])
      precision.append(test_metrics[2])
      recall.append(test_metrics[3])
      auc.append(test_metrics[4])

print(f"Loss::{np.mean(loss)}")
print(f"Accuracy::{np.mean(accuracy)}")
print(f"Precision::{np.mean(precision)}")
print(f"Recall::{np.mean(recall)}")
print(f"AUC::{np.mean(auc)}")

Test Batch 0 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.0960 - precision: 1.0000 - recall: 1.0000
Test Batch 1 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.2710 - precision: 0.7500 - recall: 1.0000
Test Batch 2 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.1989 - precision: 0.8333 - recall: 1.0000
Test Batch 3 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - AUC: 0.7333 - accuracy: 0.6250 - loss: 0.7442 - precision: 0.6667 - recall: 0.8000
Test Batch 4 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - AUC: 1.0000 - accuracy: 0.7500 - loss: 0.4827 - precision: 0.7143 - recall: 1.0000
Test Batch 5 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.1501 - precision: 0.8333 - recall: 1.0000
Loss::0.3238237996896108
Accuracy::0.8333333333333334
Precision::0.9555555582046509
Recall::0.7996031741301218
AUC::0.9666666686534882


In [29]:
f1 = 2*(0.955*0.799)/(0.955+0.799)
f1

0.8700627137970354

In [28]:
loss, accuracy, precision, recall, auc = list(), list(), list(), list(), list()
for i in range(0,6):
      test_X0 = np.load(test_dir+f"Batch{i}X0.npy")
      test_X1 = np.load(test_dir+f"Batch{i}X1.npy")
      test_X2 = np.load(test_dir+f"Batch{i}X2.npy")
      test_y = np.load(test_dir+f"Batch{i}y.npy")
      print(f"Test Batch {i} of 6")
      test_metrics = model2.evaluate([test_X0, test_X1], test_y)
      loss.append(test_metrics[0])
      accuracy.append(test_metrics[1])
      precision.append(test_metrics[2])
      recall.append(test_metrics[3])
      auc.append(test_metrics[4])

print(f"Loss::{np.mean(loss)}")
print(f"Accuracy::{np.mean(accuracy)}")
print(f"Precision::{np.mean(precision)}")
print(f"Recall::{np.mean(recall)}")
print(f"AUC::{np.mean(auc)}")

Test Batch 0 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.2925 - precision: 1.0000 - recall: 1.0000
Test Batch 1 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.3361 - precision: 1.0000 - recall: 1.0000
Test Batch 2 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.3100 - precision: 1.0000 - recall: 1.0000
Test Batch 3 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.3407 - precision: 0.8333 - recall: 1.0000
Test Batch 4 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - AUC: 1.0000 - accuracy: 0.7500 - loss: 0.4094 - precision: 0.7143 - recall: 1.0000
Test Batch 5 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - AUC: 0.8667 - accuracy: 0.8750 - loss: 0.4221 - precision: 0.8333 - recall: 1.0000
Loss::0.3517926086982091
Accuracy::0.9166666666666666
Precision::0.9777777791023254
Recall::0.896825393040975
AUC::1.0


In [30]:
f1 = 2*(0.977*0.897)/(0.977+0.897)
f1

0.9352924226254001

In [21]:
loss, accuracy, precision, recall, auc = list(), list(), list(), list(), list()
for i in range(0,6):
      test_X0 = np.load(test_dir+f"Batch{i}X0.npy")
      test_X1 = np.load(test_dir+f"Batch{i}X1.npy")
      test_X2 = np.load(test_dir+f"Batch{i}X2.npy")
      test_y = np.load(test_dir+f"Batch{i}y.npy")
      print(f"Test Batch {i} of 6")
      test_metrics = model3.evaluate([test_X0, test_X1, test_X2], test_y)
      loss.append(test_metrics[0])
      accuracy.append(test_metrics[1])
      precision.append(test_metrics[2])
      recall.append(test_metrics[3])
      auc.append(test_metrics[4])

print(f"Loss::{np.mean(loss)}")
print(f"Accuracy::{np.mean(accuracy)}")
print(f"Precision::{np.mean(precision)}")
print(f"Recall::{np.mean(recall)}")
print(f"AUC::{np.mean(auc)}")

Test Batch 0 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 19s 19s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.3587 - precision: 1.0000 - recall: 1.0000
Test Batch 1 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.2877 - precision: 1.0000 - recall: 1.0000
Test Batch 2 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.2897 - precision: 1.0000 - recall: 1.0000
Test Batch 3 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.3881 - precision: 0.8333 - recall: 1.0000
Test Batch 4 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - AUC: 1.0000 - accuracy: 0.7500 - loss: 0.4004 - precision: 0.7143 - recall: 1.0000
Test Batch 5 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - AUC: 0.6667 - accuracy: 0.8750 - loss: 0.4227 - precision: 0.8333 - recall: 1.0000
Loss::0.3578712244828542
Accuracy::0.9166666666666666
Precision::0.9444444378217062
Recall::0.896825393040975
AUC::1.0
